In [1]:
import os
from pathlib import Path

import polars as pl
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import mllabs

data_path = Path('data')

In [2]:
from mllabs.processor import PolarsLoader
ploader = PolarsLoader(predefined_types={'id': pl.Int64, 'obj_id': pl.Float64}, infer_schema_length=10000)
ploader.fit([data_path / 'train.csv', data_path / 'test.csv', data_path / 'star_classification.csv'])
df_train = ploader.transform([data_path / 'train.csv'])
df_test = ploader.transform([data_path / 'test.csv']).with_columns(
   pl.lit('').alias('class')
)

In [3]:
from itertools import combinations
from mllabs.processor import ExprProcessor
expr_dict1 = {
    'u': pl.when(pl.col('u') < 10).then(
        pl.when(pl.col("class") == "STAR").then(pl.col("u")).otherwise(None).mean()
    ).otherwise(pl.col('u')),
    'alpha90': pl.col('alpha') + 90,
    'spectral_type_galaxy_population': (pl.col('spectral_type').cast(pl.String) + '_' + pl.col('galaxy_population').cast(pl.String)).cast(pl.Categorical)
}
X_mags = ['u', 'g', 'r', 'i', 'z']
expr_dict2 = {
    'mag_mean': pl.mean_horizontal(*X_mags),
    'mag_std': pl.concat_list(X_mags).list.std(),
    'mag_min': pl.min_horizontal(*X_mags),
    'mag_max': pl.max_horizontal(*X_mags),
    'mag_range': pl.max_horizontal(*X_mags) - pl.min_horizontal(*X_mags),
}
X_mags_stat = list(expr_dict2.keys())
expr_dict2 = {
    **expr_dict2,
    'mag_vmax': pl.struct(X_mags).map_elements(
        lambda x: max(x, key=x.get),
        return_dtype=pl.String),
    'redshift_log': (pl.col('redshift') + 1e-1).log(),
    'redshift_1e-4': (pl.col('redshift') == 0.0001).cast(pl.Int8),
    'spectral_type_ord': pl.col('spectral_type').replace({'M': 0, 'G/K': 1, 'A/F': 2, 'O/B': 3}).to_physical().cast(pl.Int8),
    'galaxy_population_i': pl.when(pl.col('galaxy_population') == 'Red_Sequence').then(1).otherwise(0).cast(pl.Int8),
}
X_diff = list()
X_diff = list()
for i, j in combinations(X_mags, 2):
    X_diff.append(f'{i}_{j}')
    expr_dict2[X_diff[-1]] = pl.col(i) - pl.col(j)

X_mags_log = list()
for i in X_mags:
    X_mags_log.append(f'{i}_log')
    expr_dict2[X_mags_log[-1]] = pl.col(i).log()

In [4]:
from sklearn.pipeline import make_pipeline
expr_p = make_pipeline(ExprProcessor(expr_dict1), ExprProcessor(expr_dict2))
df_train = expr_p.fit_transform(df_train)
df_test = expr_p.transform(df_test)

In [5]:
import pickle as pkl
if not os.path.exists('data/lof.pkl'):
    from sklearn.neighbors import LocalOutlierFactor
    from sklearn.cluster import KMeans   
    df_lof = pl.concat([df_train[['alpha90', 'delta']], df_test[['alpha90', 'delta']]])
    lof = LocalOutlierFactor()
    lof.fit(df_lof)
    lof_ = lof.negative_outlier_factor_
    clu_kmeans = KMeans(3000)
    clu_kmeans.fit(df_lof)
    km3000 = clu_kmeans.labels_
    with open('data/lof.pkl', 'wb') as f:
        pkl.dump((lof_, km3000), f)
else:
    with open('data/lof.pkl', 'rb') as f:
        lof_, km3000 = pkl.load(f)
    
df_train = df_train.with_columns(
    pl.Series('lof', lof_[:len(df_train)]),
    pl.Series('km3000', km3000[:len(df_train)], dtype=pl.String).cast(pl.Categorical)
)
df_test = df_test.with_columns(
    pl.Series('lof', lof_[len(df_train):]),
    pl.Series('km3000', km3000[len(df_train):], dtype=pl.String).cast(pl.Categorical)
)
df_train.head()

id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class,alpha90,spectral_type_galaxy_population,mag_mean,mag_std,mag_min,mag_max,mag_range,mag_vmax,redshift_log,redshift_1e-4,spectral_type_ord,galaxy_population_i,u_g,u_r,u_i,u_z,g_r,g_i,g_z,r_i,r_z,i_z,u_log,g_log,r_log,i_log,z_log,lof,km3000
i64,f32,f32,f32,f32,f32,f32,f32,f32,cat,cat,cat,f32,cat,f32,f32,f32,f32,f32,str,f32,i8,i8,i8,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,cat
0,147.734253,16.959272,25.472122,21.895559,20.357925,19.257113,18.621058,0.408982,"""M""","""Red_Sequence""","""GALAXY""",237.734253,"""M_Red_Sequence""",21.120754,2.731221,18.621058,25.472122,6.851065,"""u""",-0.675342,0,0,1,3.576563,5.114197,6.21501,6.851065,1.537634,2.638447,3.274502,1.100813,1.736868,0.636055,3.237585,3.086284,3.01347,2.957881,2.924293,-1.265492,"""298"""
1,127.988678,32.346718,20.778509,19.087063,17.587208,17.226067,16.786432,0.157976,"""M""","""Red_Sequence""","""GALAXY""",217.988678,"""M_Red_Sequence""",18.293055,1.636653,16.786432,20.778509,3.992077,"""u""",-1.35489,0,0,1,1.691446,3.191301,3.552443,3.992077,1.499855,1.860996,2.300631,0.361141,0.800776,0.439634,3.03392,2.949011,2.867172,2.846424,2.820571,-1.085606,"""745"""
2,179.792648,35.344845,21.035202,21.079128,21.171841,20.58263,20.557365,2.82377,"""O/B""","""Blue_Cloud""","""QSO""",269.792664,"""O/B_Blue_Cloud""",20.885233,0.292103,20.557365,21.171841,0.614475,"""r""",1.072874,0,3,0,-0.043926,-0.136639,0.452572,0.477837,-0.092712,0.496498,0.521763,0.589211,0.614475,0.025265,3.046198,3.048284,3.052672,3.024448,3.02322,-1.186385,"""2279"""
3,225.818298,48.56942,23.305056,21.050735,19.017754,18.365658,17.914951,0.536099,"""M""","""Red_Sequence""","""GALAXY""",315.818298,"""M_Red_Sequence""",19.93083,2.235331,17.914951,23.305056,5.390104,"""u""",-0.452402,0,0,1,2.25432,4.287302,4.939398,5.390104,2.032982,2.685078,3.135784,0.652096,1.102802,0.450706,3.148671,3.046936,2.945373,2.910483,2.885636,-0.991541,"""560"""
4,141.836136,19.342852,21.703157,19.47168,18.234449,17.899446,17.616184,0.555761,"""M""","""Red_Sequence""","""GALAXY""",231.836136,"""M_Red_Sequence""",18.984983,1.676354,17.616184,21.703157,4.086973,"""u""",-0.421958,0,0,1,2.231478,3.468708,3.803711,4.086973,1.23723,1.572233,1.855495,0.335003,0.618265,0.283262,3.077458,2.968961,2.903313,2.88477,2.868818,-1.048082,"""2937"""


In [6]:
y2 = 'class'
class_weight = df_train[y2].to_pandas().value_counts().pipe(
    lambda x: x / x.min()
).to_dict()
class_weight

{'GALAXY': 4.56312557419854, 'QSO': 1.4160703060780426, 'STAR': 1.0}

In [7]:
y = 'class_i'
y_repl = {'STAR': 0, 'QSO': 1, 'GALAXY': 2}
y_weight = {'Low': 1.000000, 'Medium': 1.547291, 'High': 17.607549}
df_train = df_train.with_columns(
    **{
        y: pl.col(y2).replace(y_repl).cast(pl.Int8),
        'sample_weight': pl.col(y2).cast(pl.String).replace(class_weight).cast(pl.Float32)
    }
)

In [8]:
X_loc = ['alpha', 'delta']
X_num = ['redshift', 'redshift_log', 'lof', 'alpha90']
X_bin = ['redshift_1e-4', 'galaxy_population_i']
X_nom = ['galaxy_population', 'spectral_type', 'spectral_type_galaxy_population']
X_base = X_loc[1:] + X_num + X_bin + X_nom[1:] + X_diff + X_mags_stat + X_mags_log

X_nom2 = ['km3000']
X_ohe = ['spectral_type', 'spectral_type_galaxy_population']
X_std = ['redshift_log', 'lof'] + X_mags + X_mags_stat + X_mags_log + X_diff

In [9]:
X_num = X_loc + X_num  + X_mags+ X_mags_stat + X_mags_log + X_diff 
X_bin = ['redshift_1e-4', 'galaxy_population_i']
X_nom = ['galaxy_population', 'spectral_type', 'spectral_type_galaxy_population', 'km3000']

In [10]:
df_train[X_std]

redshift_log,lof,u,g,r,i,z,mag_mean,mag_std,mag_min,mag_max,mag_range,u_log,g_log,r_log,i_log,z_log,u_g,u_r,u_i,u_z,g_r,g_i,g_z,r_i,r_z,i_z
f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
-0.675342,-1.265492,25.472122,21.895559,20.357925,19.257113,18.621058,21.120754,2.731221,18.621058,25.472122,6.851065,3.237585,3.086284,3.01347,2.957881,2.924293,3.576563,5.114197,6.21501,6.851065,1.537634,2.638447,3.274502,1.100813,1.736868,0.636055
-1.35489,-1.085606,20.778509,19.087063,17.587208,17.226067,16.786432,18.293055,1.636653,16.786432,20.778509,3.992077,3.03392,2.949011,2.867172,2.846424,2.820571,1.691446,3.191301,3.552443,3.992077,1.499855,1.860996,2.300631,0.361141,0.800776,0.439634
1.072874,-1.186385,21.035202,21.079128,21.171841,20.58263,20.557365,20.885233,0.292103,20.557365,21.171841,0.614475,3.046198,3.048284,3.052672,3.024448,3.02322,-0.043926,-0.136639,0.452572,0.477837,-0.092712,0.496498,0.521763,0.589211,0.614475,0.025265
-0.452402,-0.991541,23.305056,21.050735,19.017754,18.365658,17.914951,19.93083,2.235331,17.914951,23.305056,5.390104,3.148671,3.046936,2.945373,2.910483,2.885636,2.25432,4.287302,4.939398,5.390104,2.032982,2.685078,3.135784,0.652096,1.102802,0.450706
-0.421958,-1.048082,21.703157,19.47168,18.234449,17.899446,17.616184,18.984983,1.676354,17.616184,21.703157,4.086973,3.077458,2.968961,2.903313,2.88477,2.868818,2.231478,3.468708,3.803711,4.086973,1.23723,1.572233,1.855495,0.335003,0.618265,0.283262
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
-0.491801,-1.009542,20.82873,18.8542,17.703108,17.190536,16.551355,18.225586,1.682177,16.551355,20.82873,4.277374,3.036334,2.936736,2.87374,2.844359,2.806468,1.974529,3.125622,3.638193,4.277374,1.151093,1.663664,2.302845,0.512571,1.151752,0.639181
-0.276295,-1.115867,23.734743,22.359173,20.697865,19.180264,18.947275,20.983866,2.057989,18.947275,23.734743,4.787468,3.16694,3.107237,3.030031,2.953882,2.94166,1.37557,3.036879,4.55448,4.787468,1.661308,3.178909,3.411898,1.517601,1.750589,0.232988
-0.741618,-1.030707,21.94425,21.215857,19.025967,18.772276,18.203396,19.832348,1.643298,18.203396,21.94425,3.740854,3.088506,3.054749,2.945805,2.932381,2.901608,0.728394,2.918283,3.171974,3.740854,2.18989,2.443581,3.012461,0.253691,0.822571,0.56888


In [11]:
X_ohe = ['spectral_type', 'spectral_type_galaxy_population']
X_std = ['redshift_log', 'lof'] + X_mags + X_mags_stat + X_mags_log + X_diff

In [12]:
from mllabs import Experimenter
from mllabs import PipelineBuilder
from mllabs import ProgressSessionLogger, TqdmProgressSession
from mllabs.filter import RandomFilter
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import balanced_accuracy_score

In [13]:
!rm -rf exp
!rm -rf exp/phase2

In [14]:
p = PipelineBuilder(path='exp', name='phase2_pipeline')

In [15]:
p.set_datasource({
    **{i: 'numerical' for i in X_num},
    **{i: 'binary' for i in X_bin},
    **{i: 'nominal' for i in X_nom+ [y, y2]},
}, targets=[y, y2])

'update'

In [16]:
def dsl_set(cols):
    return '{' + ', '.join(cols) + '}'

y_edges = {'y': dsl_set([y])}

p.set_grp('pre', role = 'stage', method='transform')
p.set_grp('pre_ft', role = 'stage', method='fit_transform', edges = y_edges)
p.set_grp('pre_ft2', role = 'stage', method='fit_transform', edges = {'y': dsl_set([y2])})

p.set_node('std', grp='pre', processor='sklearn.preprocessing.StandardScaler', edges={'X': dsl_set(X_std)})
p.set_node('ohe', grp='pre', processor='sklearn.preprocessing.OneHotEncoder', edges={'X': dsl_set(X_ohe)}, params={'sparse_output': False})
p.set_node('coov', processor = 'mllabs.processor.CatOOVFilter', grp = 'pre', edges = {'X': dsl_set(X_nom)})
p.set_grp('clf', method = 'predict', role = 'head', edges = y_edges)
p.set_grp(
    'xgb', parent = 'clf', processor='xgboost.XGBClassifier', adapter={'__ref__': 'mllabs.adapter.XGBoostAdapter', '__params__': {'eval_mode': 'both'}}, 
    params={'random_state': 123, 'n_estimators': 10000, 'enable_categorical': True, 'early_stopping_rounds': 50, 'eval_metric': 'mlogloss'})
p.set_grp(
    'lgb', parent = 'clf', processor='lightgbm.LGBMClassifier', adapter={'__ref__': 'mllabs.adapter.LightGBMAdapter', '__params__': {'eval_mode': 'both'}}, 
    params={'random_state': 123, 'n_estimators': 10000, 'verbose': -1, 'early_stopping': {'stopping_rounds': 50, 'first_metric_only': True},'eval_metric': 'multi_logloss'}
)
p.set_grp(
    'cb', parent = 'clf', processor='catboost.CatBoostClassifier', adapter={'__ref__': 'mllabs.adapter.CatBoostAdapter', '__params__': {'eval_mode': 'valid'}},
    params={'early_stopping_rounds': 50, 'eval_metric': 'AUC', 'verbose': 0, 'random_state': 123, 
            'cat_features': {'__ref__': 'mllabs.ColSelector', '__params__': {'dsl_string': '*@categorical'}}}
)
p.set_grp('nn', parent = 'clf', processor = 'mllabs.nn.NNClassifier', params = {'metrics': ['sparse_categorical_crossentropy'], 'early_stopping': 10})
p.set_grp('lr', parent='clf', processor='sklearn.linear_model.LogisticRegression')
p.set_grp('dt', parent='clf', processor='sklearn.tree.DecisionTreeClassifier', params={'random_state': 123})
p.set_node('tgt_km3000', grp = 'pre_ft2', processor='sklearn.preprocessing.TargetEncoder', edges = {'X': dsl_set(X_nom2)}, params = {'target_type': 'multiclass'})
p.set_node('xgb1', grp='xgb', edges={'X': dsl_set(X_num) + ' + coov:(*)'}, 
           params={'n_estimators': 10000})
p.set_node('lgb1', grp='lgb', edges={'X': dsl_set(X_base)}, params={'n_estimators': 10000, 'gpu': None})
p.set_node('cb1', grp='cb', edges={'X': dsl_set(X_base)}, params={'n_estimators': 10000})
p.set_node('nn1', grp='nn', edges={'X': 'std:(*) + ohe:(*@ohe_drop_first)'}, params={'epochs': 200})
p.set_node('lr1', grp='lr', edges={'X': 'std:(*) + ohe:(*@ohe_drop_first)'})

{'result': 'new',
 'affected_nodes': [],
 'old_obj': None,
 'obj': <mllabs._pipeline.PipelineNode at 0x7ff90a1f2a20>}

In [17]:
from IPython.display import display, Markdown
display(Markdown(p.desc_node('cb1')))

```mermaid
graph TD

    DataSource([DataSource])
    style DataSource fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_cb1["clf/cb/cb1"]
        cb1_dummy[ ]
        style cb1_dummy fill:none,stroke:none
    end
    style node_cb1 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    DataSource -->|X,y| node_cb1
```

**Path from DataSource to 'clf/cb/cb1' (1 path(s) found)**

### Edges

| Key | Node | Var |
|-----|------|-----|
| X | Data Source | `{delta, redshift, redshift_log, lof, alpha90, redshift_1e-4, galaxy_population_i, spectral_type, spectral_type_galaxy_population, u_g, u_r, u_i, u_z, g_r, g_i, g_z, r_i, r_z, i_z, mag_mean, mag_std, mag_min, mag_max, mag_range, u_log, g_log, r_log, i_log, z_log}` |
| y | Data Source | `{class_i}` |

In [18]:
display(Markdown(p.desc_pipeline()))

```mermaid
graph TD

    DataSource([DataSource])
    style DataSource fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_pre["pre"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_coov["coov"]
        style node_coov fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_pre fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_pre_ft2["pre_ft2"]
        node_tgt_km3000["tgt_km3000"]
        style node_tgt_km3000 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_pre_ft2 fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp___datasource__["__datasource__"]
    end
    style grp___datasource__ fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_pre_ft["pre_ft"]
    end
    style grp_pre_ft fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_xgb["xgb"]
            node_xgb1["xgb1"]
            style node_xgb1 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        end
        style grp_xgb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_lgb["lgb"]
            node_lgb1["lgb1"]
            style node_lgb1 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        end
        style grp_lgb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_cb["cb"]
            node_cb1["cb1"]
            style node_cb1 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        end
        style grp_cb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_nn["nn"]
            node_nn1["nn1"]
            style node_nn1 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        end
        style grp_nn fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_lr["lr"]
            node_lr1["lr1"]
            style node_lr1 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_dt["dt"]
        end
        style grp_dt fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    DataSource --> grp_clf
    DataSource --> grp_pre
    DataSource --> grp_pre_ft2
    grp_pre --> grp_clf
```

In [19]:
logger = ProgressSessionLogger(level=['info', 'progress'], session_cls=TqdmProgressSession)

if not os.path.exists('exp/phase2'):
    sss1 = StratifiedShuffleSplit(n_splits = 1, random_state = 123, train_size = 0.8)
    sss2 = StratifiedShuffleSplit(n_splits = 1, random_state = 123, train_size = 0.9)
    e = Experimenter(df_train, sp = sss1, sp_v = sss2, splitter_params={'y': y}, path='exp/phase2')
    with e.os_log():
        e.set_collector('xgb_evals_results', 'mllabs.collector.ModelAttrCollector', {'__ref__': 'mllabs.Connector', '__params__': {'processor': 'xgboost.XGBClassifier'}}, params={'result_key': 'evals_result'})
        e.set_collector('lgb_evals_results', 'mllabs.collector.ModelAttrCollector', {'__ref__': 'mllabs.Connector', '__params__': {'processor': 'lightgbm.LGBMClassifier'}}, params={'result_key': 'evals_result'})
        e.set_collector('cb_evals_results', 'mllabs.collector.ModelAttrCollector', {'__ref__': 'mllabs.Connector', '__params__': {'processor': 'catboost.CatBoostClassifier'}}, params={'result_key': 'evals_result'})
        e.set_collector('nn_evals', 'mllabs.collector.ModelAttrCollector', {'__ref__': 'mllabs.Connector', '__params__': {'processor': 'mllabs.nn.NNClassifier'}}, params={'result_key': 'evals_result'})
        e.set_collector('bAcc', 'mllabs.collector.MetricCollector', {'__ref__': 'mllabs.Connector', '__params__': {'edges': y_edges, 'role': 'head'}}, params={'output_var': '-1:', 'metric_func': {'__callable__': 'sklearn.metrics.balanced_accuracy_score'}, 'include_train': True})
        e.set_collector('lgb_feature_importance', 'mllabs.collector.ModelAttrCollector', {'__ref__': 'mllabs.Connector', '__params__': {'processor': 'lightgbm.LGBMClassifier', 'edges': y_edges}}, params={'result_key': 'feature_importances'})
        e.set_collector('xgb_feature_importance_gain', 'mllabs.collector.ModelAttrCollector', {'__ref__': 'mllabs.Connector', '__params__': {'processor': 'xgboost.XGBClassifier', 'edges': y_edges}}, params={'result_key': 'feature_importances', 'params': {'importance_type': 'gain'}})
        e.set_collector('xgb_feature_importance_cover', 'mllabs.collector.ModelAttrCollector', {'__ref__': 'mllabs.Connector', '__params__': {'processor': 'xgboost.XGBClassifier', 'edges': y_edges}}, params={'result_key': 'feature_importances', 'params': {'importance_type': 'cover'}})
        e.set_collector('cb_feature_importance', 'mllabs.collector.ModelAttrCollector', {'__ref__': 'mllabs.Connector', '__params__': {'processor': 'catboost.CatBoostClassifier', 'edges': y_edges}}, params={'result_key': 'feature_importances_pvc'})
        e.set_collector('cb_interaction', 'mllabs.collector.ModelAttrCollector', {'__ref__': 'mllabs.Connector', '__params__': {'processor': 'catboost.CatBoostClassifier', 'edges': y_edges}}, params={'result_key': 'feature_importances_interaction'})
        e.set_collector('lr_coef', 'mllabs.collector.ModelAttrCollector', {'__ref__': 'mllabs.Connector', '__params__': {'processor': 'sklearn.linear_model.LogisticRegression', 'edges': y_edges}}, params={'result_key': 'coef'})
else:
    e = Experimenter.load('exp/phase2', df_train)
e.set_pipeline(p.build())

In [20]:
with e.os_log():
    e.build()

Building 4 node(s)
Build complete: 4 node(s)


In [21]:
with e.os_log():
    e.exp(nodes = 'xgb1', n_jobs=2, gpu_id_list=[0], logger = logger)

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Experimenting 1 node(s)
Exp complete: 1 node(s)


In [22]:
with e.os_log():
    p.set_node('xgb2', grp='xgb', edges={'X': dsl_set(X_num + X_bin) + ' + coov:(*) + tgt_km3000:(*)'}, 
               params={'n_estimators': 10000})
    p.set_node('lgb2', grp='lgb', edges={'X': dsl_set(X_base) + ' + tgt_km3000:(*)'}, params={'n_estimators': 10000, 'gpu': None})
    p.set_node('cb2', grp='cb', edges={'X': dsl_set(X_base) + ' + tgt_km3000:(*)'}, params={'n_estimators': 10000})
    p.set_node('nn2', grp='nn', edges={'X': 'std:(*) + ohe:(@ohe_drop_first()) + tgt_km3000:(*)'}, params={'epochs': 200})
    p.set_node('lr2', grp='lr', edges={'X': 'std:(*) + ohe:(@ohe_drop_first()) + tgt_km3000:(*)'})

In [23]:
with e.os_log():
    e.exp(n_jobs=2, gpu_id_list=[0], logger = logger)

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Experimenting 9 node(s)
Exp complete: 9 node(s)


In [24]:
e.get_collector('lgb_feature_importance').get_attrs_agg('lgb2').sort_values(ascending = False).iloc[:10]

delta                        711.0
redshift                     642.0
redshift_log                 599.0
alpha90                      565.0
tgt_km3000__km3000_GALAXY    402.0
g_z                          392.0
tgt_km3000__km3000_STAR      366.0
u_g                          354.0
g_log                        328.0
r_log                        302.0
dtype: float64

In [25]:
e.get_collector('xgb_feature_importance_gain').get_attrs_agg('xgb2').sort_values(ascending = False).iloc[:10]

g_z                          712.372070
redshift                     412.420349
redshift_1e-4                203.272980
mag_std                      145.589752
u_i                          138.424271
redshift_log                 102.520782
tgt_km3000__km3000_GALAXY     97.458145
r_log                         90.066200
g_i                           83.365303
g_log                         79.468628
dtype: float64

In [26]:
e.get_collector('lr_coef').get_attrs_agg('lr2').sort_values(ascending = False).iloc[:10]

2  std__u_log                 6.749289
1  std__r_log                 6.302431
   std__z_log                 6.010033
   std__i_log                 5.737730
   std__g_log                 4.319041
0  std__z_log                 3.530226
2  std__z                     2.463602
   std__i                     2.384900
0  tgt_km3000__km3000_STAR    2.207027
2  std__r                     2.188485
dtype: float64

In [27]:
e.get_collector('cb_interaction').get_attrs_agg('cb2').sort_values(ascending = False).iloc[:20]

feat1     feat2                    
redshift  g_z                          2.047381
delta     alpha90                      1.684129
redshift  tgt_km3000__km3000_GALAXY    1.554978
          redshift_1e-4                1.511593
delta     redshift                     1.371030
redshift  tgt_km3000__km3000_STAR      1.278351
          g_log                        1.193764
          r_z                          1.155714
          i_log                        1.012906
          z_log                        1.001599
          g_i                          0.991410
          alpha90                      0.974294
          u_i                          0.955342
          mag_min                      0.928873
delta     tgt_km3000__km3000_GALAXY    0.869071
redshift  r_log                        0.856207
          mag_mean                     0.700638
          mag_max                      0.675637
          u_g                          0.648713
          u_log                        0.639866
dtyp

In [28]:
with e.os_log():
    p.set_node('delta_100', grp = 'pre', processor = "sklearn.preprocessing.KBinsDiscretizer", edges = {'X': dsl_set(['delta'])}, params = {'n_bins': 100, 'encode': 'ordinal'})
    p.set_node('delta_500', grp = 'pre', processor = "sklearn.preprocessing.KBinsDiscretizer", edges = {'X': dsl_set(['delta'])}, params = {'n_bins': 500, 'encode': 'ordinal'})
    p.set_node('tgt_delta', grp = 'pre_ft2', processor = "sklearn.preprocessing.TargetEncoder", edges = {'X': 'delta_100:(*) + delta_500:(*)'}, params = {'target_type': 'multiclass'})
    e.build()

Building 3 node(s)
Build complete: 3 node(s)


In [30]:
with e.os_log():
    p.set_node('xgb3', grp='xgb', edges={'X': dsl_set(X_num + X_bin) + ' + coov:(*) + tgt_km3000:(*) + tgt_delta:(*)'}, 
               params={'n_estimators': 10000})
    p.set_node('lgb3', grp='lgb', edges={'X': dsl_set(X_base) + ' + tgt_km3000:(*) + tgt_delta:(*)'}, params={'n_estimators': 10000, 'gpu': None})
    p.set_node('cb3', grp='cb', edges={'X': dsl_set(X_base) + ' + tgt_km3000:(*) + tgt_delta:(*)'}, params={'n_estimators': 10000})
    p.set_node('nn3', grp='nn', edges={'X': 'std:(*) + ohe:(@ohe_drop_first()) + tgt_km3000:(*) + tgt_delta:(*)'}, params={'epochs': 200})
    p.set_node('lr3', grp='lr', edges={'X': 'std:(*) + ohe:(@ohe_drop_first()) + tgt_km3000:(*) + tgt_delta:(*)'})
    e.exp(n_jobs=2, gpu_id_list=[0], finalize=True, logger = logger)

Experimenting 1 node(s)


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Exp complete: 1 node(s)


In [32]:
with e.os_log():
    p.set_node('alpha_100', grp = 'pre', processor = "sklearn.preprocessing.KBinsDiscretizer", edges = {'X': dsl_set(['alpha90'])}, params = {'n_bins': 100, 'encode': 'ordinal'})
    p.set_node('alpha_500', grp = 'pre', processor = "sklearn.preprocessing.KBinsDiscretizer", edges = {'X': dsl_set(['alpha90'])}, params = {'n_bins': 500, 'encode': 'ordinal'})
    p.set_node('tgt_alpha', grp = 'pre_ft2', processor="sklearn.preprocessing.TargetEncoder", edges = {'X': 'alpha_100:(*) + alpha_500:(*)'}, params = {'target_type': 'multiclass'})
    e.build()

Building 3 node(s)
Build complete: 3 node(s)


In [33]:
with e.os_log():
    p.set_node('xgb4', grp='xgb', edges={'X': dsl_set(X_num + X_bin) + ' + coov:(*) + tgt_km3000:(*) + tgt_delta:(*) + tgt_alpha:(*)'}, 
               params={'n_estimators': 10000})
    p.set_node('lgb4', grp='lgb', edges={'X': dsl_set(X_base) + ' + tgt_km3000:(*) + tgt_delta:(*) + tgt_alpha:(*)'}, 
               params={'n_estimators': 10000, 'gpu': None})
    p.set_node('cb4', grp='cb', edges={'X': dsl_set(X_base) + ' + tgt_km3000:(*) + tgt_delta:(*) + tgt_alpha:(*)'}, 
               params={'n_estimators': 10000})
    p.set_node('nn4', grp='nn', edges={'X': 'std:(*) + ohe:(@ohe_drop_first()) + tgt_km3000:(*) + tgt_delta:(*) + tgt_alpha:(*)'}, 
               params={'epochs': 200})
    p.set_node('lr4', grp='lr', edges={'X': 'std:(*) + ohe:(@ohe_drop_first()) + tgt_km3000:(*) + tgt_delta:(*) + tgt_alpha:(*)'})
    e.exp(n_jobs=2, gpu_id_list=[0], finalize=True)

Experimenting 5 node(s)
Exp complete: 5 node(s)


In [34]:
e.get_collector('bAcc').get_metrics_agg(None)[0].sort_values('test', ascending = False).iloc[:15]

,test,train,valid
cb4,0.957883,0.962309,0.956453
cb2,0.957448,0.963226,0.956706
lgb4,0.956823,0.961669,0.955891
lgb3,0.956173,0.961427,0.956883
lgb2,0.956101,0.960755,0.956164
xgb4,0.954808,0.972564,0.953398
xgb2,0.953772,0.970691,0.952034
nn4,0.952525,0.954024,0.951070
cb1,0.951579,0.957281,0.950872
xgb1,0.951329,0.974642,0.949980


In [35]:
with e.os_log():
    p.set_node('cat0', grp='pre', edges={'X': 'alpha_500:(*) + delta_500:(*)'}, processor = 'mllabs.processor.CatConverter')
    p.set_node('cat1', grp='pre', edges={'X': 'alpha_100:(*) + delta_100:(*)'}, processor = 'mllabs.processor.CatConverter')
    p.set_node('lda_mags', grp = 'pre_ft2', processor='sklearn.discriminant_analysis.LinearDiscriminantAnalysis', edges = {'X': dsl_set(X_mags)})
    e.build()

Building 3 node(s)
Build complete: 3 node(s)


In [40]:
with e.os_log():
    p.set_node('xgb5', grp='xgb', edges={'X': dsl_set(X_num + X_bin) + ' + coov:(*) + tgt_km3000:(*) + tgt_delta:(*) + tgt_alpha:(*) + lda_mags:(*)'}, 
               params={'n_estimators': 10000})
    p.set_node('lgb5', grp='lgb', edges={'X': dsl_set(X_base) + ' + tgt_km3000:(*) + tgt_delta:(*) + tgt_alpha:(*) + lda_mags:(*)'}, 
               params={'n_estimators': 10000, 'gpu': None})
    p.set_node('cb5', grp='cb', edges={'X': dsl_set(X_base) + ' + tgt_km3000:(*) + tgt_delta:(*) + tgt_alpha:(*) + lda_mags:(*)'}, 
               params={'n_estimators': 10000})
    p.set_node('nn5', grp='nn', edges={'X': 'std:(*) + ohe:(@ohe_drop_first()) + tgt_km3000:(*) + tgt_delta:(*) + tgt_alpha:(*) + lda_mags:(*)'}, 
               params={'epochs': 200})
    p.set_node('nn6', grp='nn', 
               edges={'X': 'std:(*) + ohe:(@ohe_drop_first()) + tgt_km3000:(*) + tgt_delta:(*) + tgt_alpha:(*) + lda_mags:(*) + ' + dsl_set(X_nom2) + ' + cat0:(*)'}, 
               params={'epochs': 200, 'cat_cols': {'__ref__': 'ColSelector', '__params__': {'dsl_string': '*@categorical'}}})
    p.set_node('nn7', grp='nn', 
               edges={'X': 'std:(*) + ohe:(@ohe_drop_first()) + tgt_km3000:(*) + tgt_delta:(*) + tgt_alpha:(*) + lda_mags:(*) + ' + dsl_set(X_nom2) + ' + cat1:(*)'}, 
               params={'epochs': 200, 'cat_cols': {'__ref__': 'ColSelector', '__params__': {'dsl_string': '*@categorical'}}})
    p.set_node('nn8', grp='nn', 
               edges={'X': 'std:(*) + ohe:(@ohe_drop_first()) + tgt_km3000:(*) + tgt_delta:(*) + tgt_alpha:(*) + lda_mags:(*) + ' + dsl_set(X_nom2)}, 
               params={'epochs': 200, 'cat_cols': {'__ref__': 'ColSelector', '__params__': {'dsl_string': '*@categorical'}}})
    p.set_node('nn9', grp='nn', 
               edges={'X': 'std:(*) + ohe:(@ohe_drop_first()) + tgt_km3000:(*) + tgt_delta:(*) + tgt_alpha:(*) + lda_mags:(*) + cat1:(*)'}, 
               params={'epochs': 200, 'cat_cols': {'__ref__': 'ColSelector', '__params__': {'dsl_string': '*@categorical'}}})
    e.exp(n_jobs=2, gpu_id_list=[0], finalize=True)

Experimenting 8 node(s)


EOFError: 

In [37]:
e.get_collector('bAcc').get_metrics_agg(None)[0].sort_values('test', ascending = False)

,test,train,valid
cb4,0.958025,0.964125,0.957791
cb5,0.957985,0.962488,0.957724
cb3,0.957761,0.963698,0.958127
lgb5,0.957630,0.961740,0.956941
lgb4,0.957248,0.961072,0.956899
lgb3,0.956957,0.960778,0.956481
cb2,0.956885,0.962289,0.956855
lgb2,0.956357,0.960422,0.955892
xgb5,0.955098,0.975765,0.955491
xgb4,0.954847,0.974940,0.955439


# dtype 기반 selector (@numeric / @categorical / @binary / @float / @int / @string)

edges DSL에서 `@numeric`/`@categorical` 등은 실제 컬럼의 dtype을 보고 선택하는 selector다 (processor 불필요).
DataSource 최상위(`*@numeric`)에 바로 걸면 `id`/`class_i`(target)/`sample_weight`처럼 스키마에 없는 raw 컬럼까지 딸려 들어올 수 있어 위험하므로,
이미 확정된 stage 노드의 출력 안에서만(namespace 안에서) 사용한다 — 여기서는 `dt`(미사용 상태였던 grp)에 데모 노드를 하나 추가한다.

In [ ]:
p.set_node('dt1', grp='dt', edges={'X': 'std:(*@numeric) + coov:(*@categorical) + tgt_km3000:(*)'})
e.exp(nodes='dt1', n_jobs=2, gpu_id_list=[0])
e.get_collector('bAcc').get_metrics_agg('dt1')[0]

In [38]:
import catboost as cb
e.set_collector('cb_evals_results', ModelAttrCollector, Connector(processor='catboost.CatBoostClassifier'), params={'result_key': 'evals_result'}, exist='replace')

In [ ]:
e.get_collect_status('cb_evals_results')

In [ ]:
e.exp()

In [55]:
e.get_collector('cb_evals_results').get_attrs_agg('cb2').unstack(level=[-1, -2])

,learn,validation,learn,validation
,AUC:type=Mu,AUC:type=Mu,MultiClass,MultiClass
0,NaN,0.986215,1.015563,1.015622
1,NaN,0.989051,0.943259,0.943297
2,NaN,0.991095,0.879686,0.879738
3,NaN,0.991244,0.823443,0.823374
4,NaN,0.991527,0.773511,0.773461
...,...,...,...,...
1881,NaN,0.997953,0.077790,0.086612
1882,NaN,0.997953,0.077784,0.086614
1883,NaN,0.997952,0.077769,0.086612


In [112]:
logger = ProgressSessionLogger(level=['info', 'progress'], session_cls=TqdmProgressSession)
if not os.path.exists('exp/phase2'):
    sss1 = StratifiedKFold(n_splits = 1, random_state = 123, train_size = 0.8)
    e = Experimenter(df_train, sp = sss1, sp_v = sss2, splitter_params={'y': y}, path='exp/phase2')
    e.set_collector('xgb_evals_results', ModelAttrCollector, Connector(processor='xgboost.XGBClassifier'), params={'result_key': 'evals_result'})
    e.set_collector('lgb_evals_results', ModelAttrCollector, Connector(processor='lightgbm.LGBMClassifier'), params={'result_key': 'evals_result'})
    e.set_collector('cb_evals_results', ModelAttrCollector, Connector('_base$', processor='catboost.CatBoostClassifier'), params={'result_key': 'evals_result'})
    e.set_collector('nn_evals', ModelAttrCollector, Connector(processor='mllabs.nn.NNClassifier'), params={'result_key': 'evals_result'})
    e.set_collector('bAcc', MetricCollector, Connector(edges = y_edges, role = 'head'), params={'output_var': slice(-1, None), 'metric_func': balanced_accuracy_score, 'include_train': True})
    e.set_collector('lgb_feature_importance', ModelAttrCollector, Connector(processor='lightgbm.LGBMClassifier', edges=y_edges), params={'result_key': 'feature_importances'})
    e.set_collector('xgb_feature_importance_gain', ModelAttrCollector, Connector(processor='xgboost.XGBClassifier', edges=y_edges), params={'result_key': 'feature_importances', 'params': {'importance_type': 'gain'}})
    e.set_collector('xgb_feature_importance_cover', ModelAttrCollector, Connector(processor='xgboost.XGBClassifier', edges=y_edges), params={'result_key': 'feature_importances', 'params': {'importance_type': 'cover'}})
    e.set_collector('cb_feature_importance', ModelAttrCollector, Connector(processor='catboost.CatBoostClassifier', edges=y_edges), params={'result_key': 'feature_importances_pvc'})
    e.set_collector('cb_interaction', ModelAttrCollector, Connector(processor='catboost.CatBoostClassifier', edges=y_edges), params={'result_key': 'feature_importances_interaction'})
    e.set_collector('lr_coef', ModelAttrCollector, Connector(processor='sklearn.linear_model.LogisticRegression', edges=y_edges), params={'result_key': 'coef'})
else:
    e = Experimenter.load('exp/phase2', df_train)

No error nodes found
